# Batch-size test: which regime is the training step in?

Times the **baseline training step** (one GPU, default AdamW, `.item()` every batch; the inner loop of `train()` in `src/smi/models/model.py`) at batch sizes 32, 64, 128 and 256, on real train batches (k = 0).

`FP16` in the setup cell switches the same step to fp16 autocast + loss scaling. Everything else is unchanged, so fp32 vs fp16 differs by that one thing.

**How to read it** (concepts/performance-analysis-and-optimization/02):
- time per step grows **proportionally** with the batch → the step is compute- or memory-bound;
- time per step grows **much less** than the batch → fixed per-step costs dominate (launch overhead, plus the optimizer and clipping, which do not depend on the batch).

Two timing modes for every batch size:
- **total:** no extra synchronisation, the real time per step;
- **breakdown:** `torch.cuda.synchronize()` between phases, the time of each phase. The syncs remove the CPU/GPU overlap, so this total is a bit higher.

An exploration notebook (PROTOCOL §8): its numbers inform decisions, they are not reportable results.

Settings: **Accelerator GPU T4** (one GPU is used), **Internet on**, the private `smi-v4-meetings` dataset attached.

## 1. GPU, code, packages, data

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/repo
!git clone -b C/baselines-design https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
!cd /kaggle/working/repo && git log --oneline -1
!pip install -q transformers==5.20.1 sentencepiece==0.2.2

In [ ]:
import glob, os, sys
sys.path.insert(0, "/kaggle/working/repo/src")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

hits = glob.glob("/kaggle/input/**/source.json", recursive=True)
assert len(hits) == 1, f"expected exactly one source.json under /kaggle/input, found {hits}"
DATA_ROOT = os.path.dirname(hits[0])
MODEL = "microsoft/deberta-v3-base"
FP16 = True  # False = the fp32 baseline step ; True = the same step under fp16 autocast + loss scaling
print("DATA_ROOT =", DATA_ROOT, "| FP16 =", FP16)

## 2. The step being measured
Exactly the baseline loop's inner step: forward → cross-entropy → backward → `loss.item()` → clip → AdamW step → scheduler step → `zero_grad`.
One model and one optimizer are reused for all batch sizes: the weights change while we measure, which does not affect speed.

In [ ]:
import time
from pathlib import Path

import torch
from transformers import get_scheduler

from smi.inputs.collator import SegmentCollator
from smi.inputs.dataset import SegmentDataset
from smi.inputs.sampler import LengthBucketBatchSampler
from smi.models.model import build_optimizer, load_model_and_tokenizer

device = torch.device("cuda")
torch.manual_seed(42)
model, tokenizer = load_model_and_tokenizer(MODEL)
model.to(device)
model.train()
optimizer = build_optimizer(model, 2.0e-5, 0.01)  # the default AdamW (not fused) , as in the baseline
scheduler = get_scheduler("linear", optimizer, num_warmup_steps=100, num_training_steps=100_000)
loss_function = torch.nn.CrossEntropyLoss()
# fp16 : multiplies the loss before backward so small fp16 gradients do not round to 0 ;
# when FP16 is False the scaler does nothing and the step is exactly the fp32 baseline
scaler = torch.amp.GradScaler("cuda", enabled=FP16)
skipped = {"steps": 0}

dataset = SegmentDataset(Path(DATA_ROOT) / "train", 0, 0)
collator = SegmentCollator(tokenizer, 512, "\n", markers=("[T] ", " [/T]"))
print(f"{len(dataset)} train segments | model on {device} | weights {model.dtype} | FP16 autocast {FP16}")


def get_batches(batch_size, count):
    """the first `count` full batches of the training sampler (same seed as the runs)."""
    sampler = LengthBucketBatchSampler(dataset, batch_size, 50, True, seed=42)
    batches = []
    for indices in sampler:
        if len(indices) == batch_size:
            batches.append(indices)
        if len(batches) == count:
            break
    return batches


def one_step(indices, breakdown, phases):
    """one baseline training step ; adds each phase's seconds to `phases` ; returns the padded length."""
    t0 = time.perf_counter()
    batch = collator([dataset[i] for i in indices])
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    if breakdown:
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    # matrix multiplications in fp16 on Tensor Cores ; weights stay fp32
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=FP16):
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    loss = loss_function(logits.float(), labels)
    if breakdown:
        torch.cuda.synchronize()
    t2 = time.perf_counter()

    scaler.scale(loss).backward()  # = loss.backward() when FP16 is False
    loss_value = loss.item()  # the baseline's GPU -> CPU sync on every batch
    t3 = time.perf_counter()

    scaler.unscale_(optimizer)  # divide the gradients by the scale again (a pass over all gradients)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    if breakdown:
        torch.cuda.synchronize()
    t4 = time.perf_counter()

    scale_before = scaler.get_scale()
    scaler.step(optimizer)  # = optimizer.step() when FP16 is False ; skipped if a gradient overflowed
    scaler.update()
    if FP16 and scaler.get_scale() < scale_before:
        skipped["steps"] += 1  # the scale is lowered exactly when a step was skipped
    scheduler.step()
    optimizer.zero_grad()
    if breakdown:
        torch.cuda.synchronize()
    t5 = time.perf_counter()

    phases["data"] += t1 - t0
    phases["forward"] += t2 - t1
    phases["backward"] += t3 - t2
    phases["clip"] += t4 - t3
    phases["optimizer"] += t5 - t4
    return input_ids.shape[1]


def measure(batch_size, warmup=10, timed=60, breakdown_steps=30):
    batches = get_batches(batch_size, warmup + timed + breakdown_steps)
    torch.cuda.reset_peak_memory_stats()
    phases = {"data": 0.0, "forward": 0.0, "backward": 0.0, "clip": 0.0, "optimizer": 0.0}

    for indices in batches[:warmup]:  # not timed : first kernels , memory allocator , caches
        one_step(indices, False, phases)

    # total mode : no extra syncs
    lengths = []
    torch.cuda.synchronize()
    start = time.perf_counter()
    for indices in batches[warmup:warmup + timed]:
        lengths.append(one_step(indices, False, phases))
    torch.cuda.synchronize()
    seconds_per_step = (time.perf_counter() - start) / timed

    # breakdown mode : a sync between phases
    for name in phases:
        phases[name] = 0.0
    for indices in batches[warmup + timed:]:
        one_step(indices, True, phases)

    padded_length = sum(lengths) / len(lengths)
    return {
        "batch": batch_size,
        "ms_per_step": 1000 * seconds_per_step,
        "padded_length": padded_length,
        "tokens_per_step": batch_size * padded_length,
        "min_per_epoch": (78903 / batch_size) * seconds_per_step / 60,
        "peak_gb": torch.cuda.max_memory_allocated() / 1e9,
        "fp16": FP16,
        "phases_ms": {name: 1000 * value / breakdown_steps for name, value in phases.items()},
    }

## 3. Run

In [ ]:
# warm-up pass , discarded : the first measurement also pays one-time costs
# (kernel selection for new tensor shapes , memory allocator growth)
for batch_size in (32, 64, 128, 256):
    try:
        measure(batch_size, warmup=10, timed=5, breakdown_steps=1)
    except torch.OutOfMemoryError:
        torch.cuda.empty_cache()
        break
print("warm-up done | fp16 skipped steps so far:", skipped["steps"], flush=True)

results = []
for batch_size in (32, 64, 128, 256):
    try:
        r = measure(batch_size)
        results.append(r)
        print(f"batch {batch_size:>3}: {r['ms_per_step']:7.1f} ms/step | {r['min_per_epoch']:5.1f} min/epoch | "
              f"padded length {r['padded_length']:5.1f} | peak {r['peak_gb']:.1f} GB", flush=True)
    except torch.OutOfMemoryError:
        print(f"batch {batch_size:>3}: out of memory", flush=True)
        torch.cuda.empty_cache()
        break

## 4. Results

In [ ]:
base = results[0]
print("precision:", "fp16 autocast" if FP16 else "fp32", "| fp16 skipped steps:", skipped["steps"])
print(f"{'batch':>5} | {'ms/step':>8} | {'x time':>6} | {'x tokens':>8} | {'ms/1k tokens':>12} | {'min/epoch':>9} | {'peak GB':>7}")
for r in results:
    print(f"{r['batch']:>5} | {r['ms_per_step']:>8.1f} | {r['ms_per_step'] / base['ms_per_step']:>6.2f} | "
          f"{r['tokens_per_step'] / base['tokens_per_step']:>8.2f} | {1000 * r['ms_per_step'] / r['tokens_per_step']:>12.2f} | "
          f"{r['min_per_epoch']:>9.1f} | {r['peak_gb']:>7.1f}")

print(f"\n{'batch':>5} | " + " | ".join(f"{name:>9}" for name in base["phases_ms"]) + " | breakdown total (ms)")
for r in results:
    p = r["phases_ms"]
    print(f"{r['batch']:>5} | " + " | ".join(f"{p[name]:>9.1f}" for name in p) + f" | {sum(p.values()):>9.1f}")

**Reading the tables**
- `x time` vs `x tokens`: if time grows much less than the tokens, fixed costs dominate.
- `clip` and `optimizer` should stay roughly constant across batch sizes: they touch all 184M parameters whatever the batch.
- `forward` and `backward` are the parts that grow with the batch. How fast they grow says whether the model itself is overhead-bound or compute-bound at these sizes.
- `data` is the CPU's batch preparation (tokenizing + copying to the GPU).